# Part 12 · Notebook 09 — Rollout, capital ramp and daily operations

**Sessions:** S17–S18 (Rollout plan & capital ramp · Daily operations) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Move a strategy through paper → shadow → small live → scaled live on evidence, and demote it on evidence.
2. Ramp capital by written rules, and cut it automatically after a drawdown.
3. Know the trading calendar: holidays and early closes.
4. Automate the pre-market checklist, so a failed blocking check stops trading.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Stages, promotion and demotion

**Paper** tests the plumbing, **shadow** runs on live data and logs orders without sending them, **small live** trades minimum size, **scaled live** the planned size. `p.rollout_decision` applies written rules:

* **Demote** one stage if fewer than half of the days were inside the backtest band, a drift alarm fired, or there were 3 or more incidents.
* **Promote** one stage only if *every* criterion holds: enough days at this stage, no sev1 incident, no open reconciliation break, slippage at most 1.5× the model, and at least 80% of the days inside the band.
* Otherwise **hold**, and say which criteria failed.

In [ ]:
base = dict(days=12, sev1_incidents=0, incidents=0, open_breaks=0, slippage_ratio=1.1, inside_band=0.9, drift_alarm=False)
cases = {"paper, 12 clean days": ("paper", base),
         "shadow, 12 clean days": ("shadow", base),
         "small live, 12 days": ("small_live", base),
         "small live, 25 days, a sev1 and 1.8× slippage": ("small_live", {**base, "days": 25, "sev1_incidents": 1, "slippage_ratio": 1.8}),
         "small live, 25 days, 40% inside the band": ("small_live", {**base, "days": 25, "inside_band": 0.4}),
         "scaled live, a drift alarm": ("scaled_live", {**base, "days": 60, "drift_alarm": True})}
pd.DataFrame({k: dict(zip(["action", "new stage", "reasons"], p.rollout_decision(*c))) for k, c in cases.items()}).T

Rules written before going live stop the two classic mistakes: scaling up after a lucky week, and "giving it one more week" while a broken strategy bleeds.

## 2. The capital ramp

Size follows a plan too (`PLAN`). A drawdown worse than `max_dd` halves the capital fraction, but not below `min`. Otherwise, every full `weeks_per_step` weeks inside the band add `step`, up to `max`: increase only when `weeks_in_band` is a positive multiple of `weeks_per_step`. Anything else: no change.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
PLAN = {"min": 0.1, "max": 1.0, "step": 0.1, "weeks_per_step": 2, "max_dd": -0.03}

def capital_ramp(fraction, weeks_in_band, drawdown, plan):
    if drawdown < plan["max_dd"]:
        return ...                                                   # ✍️ halve, but not below plan["min"]
    if ...:                                                          # ✍️ a positive multiple of plan["weeks_per_step"]
        return min(fraction + plan["step"], plan["max"])
    return fraction

cases = [(0.1, 2, -0.01), (0.1, 3, -0.01), (0.5, 4, -0.02), (0.4, 6, -0.05), (0.15, 0, -0.04), (1.0, 8, 0.0), (0.2, 0, 0.0)]
mine = p.check("capital_ramp", [p.attempt(capital_ramp, *c, PLAN) for c in cases], [p.capital_ramp(*c, PLAN) for c in cases])
pd.DataFrame(cases, columns=["fraction", "weeks in band", "drawdown"]).assign(next_week=mine)

In [ ]:
weeks = pd.DataFrame({"in_band": [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                      "drawdown": [0, 0, -0.01, 0, 0, -0.01, -0.045, -0.02, -0.01, 0, 0, 0, 0, -0.01, 0, 0]}, index=range(1, 17))
frac, streak, path = 0.1, 0, []
for w, row in weeks.iterrows():
    streak = streak + 1 if row["in_band"] else 0
    frac = p.capital_ramp(frac, streak, row["drawdown"], PLAN)
    path.append(frac)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.step(weeks.index, path, where="post")
ax.axvline(7, color=p.PALETTE[7], ls="--", lw=1, label="week 7: outside the band, drawdown −4.5%")
ax.set(title="Capital fraction under the ramp rules", xlabel="week", ylabel="share of planned capital")
ax.legend(loc="upper left")
plt.show()
print("fraction by week:", [round(f, 2) for f in path])

Slow up, fast down: two weeks in the band for each step up; one bad week halves the size and resets the count.

## 3. The trading calendar

The session is 09:30–16:00 New York time, or 09:30–13:00 on early-close days (`p.EARLY_CLOSE`); there is no session on weekends and exchange holidays (`p.HOLIDAYS`). A strategy that assumes 16:00 on the day after Thanksgiving sends its closing orders three hours after the market closed.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def session_times(day):
    d = pd.Timestamp(day).normalize()
    if ...:                                                          # ✍️ a weekend (weekday() >= 5) or a holiday
        return None
    close = pd.Timedelta(hours=13) if d in p.EARLY_CLOSE else pd.Timedelta(hours=16)
    return d + pd.Timedelta(hours=9, minutes=30), d + close

days = [*pd.date_range("2025-07-02", "2025-07-07"), *pd.date_range("2025-11-26", "2025-11-28"), pd.Timestamp("2025-12-24")]
mine = p.check("session_times", {str(d.date()): p.attempt(session_times, d) for d in days},
               {str(d.date()): p.session_times(d) for d in days})
pd.DataFrame({day: {"open": t[0].time() if t else None, "close": t[1].time() if t else None} for day, t in mine.items()}).T

## 4. The pre-market checklist

Every morning, before the open, n8n triggers a script that runs every check in `p.PREMARKET`. Failing any check in `p.BLOCKING` (gateway connected, data fresh, reconciliation clean, kill switch tested today, risk limits loaded) blocks trading; the others (backup age, free disk) are warnings. Return the results, the sorted blocking failures, the sorted warnings, and whether trading may start.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def premarket_check(status):
    res = {k: bool(f(status)) for k, f in p.PREMARKET.items()}
    failed = {k for k, ok in res.items() if not ok}
    return {"results": res, "blocking_failures": ..., "warnings": ..., "may_trade": ...}   # ✍️

good = {"gateway_connected": True, "data_age_s": 1.2, "open_breaks": 0, "kill_switch_tested_today": True,
        "risk_limits_loaded": True, "last_backup_age_h": 20, "disk_free_pct": 40}
cases = {"a normal morning": good,
         "after a long weekend": {**good, "kill_switch_tested_today": False, "last_backup_age_h": 50},
         "feed down, disk filling": {**good, "data_age_s": 300, "disk_free_pct": 9}}
mine = p.check("premarket_check", {k: p.attempt(premarket_check, s) for k, s in cases.items()},
               {k: p.premarket_check(s) for k, s in cases.items()})
pd.DataFrame({k: {c: v[c] for c in ("blocking_failures", "warnings", "may_trade")} for k, v in mine.items()}).T

An untested kill switch is not a kill switch, so that check blocks; an old backup is serious but does not make the day's trading unsafe, so it warns. **Intraday:** watch the dashboards, handle alerts, no ad-hoc parameter changes. **End of day:** reconcile, review the journal, send the daily report, confirm the backup, log incidents.

## Wrap-up

* Stages with written promotion and demotion rules, applied the same way every time.
* Capital ramps slowly and is cut automatically.
* The calendar knows holidays and early closes.
* The pre-market script's blocking failures stop trading; warnings get fixed the same day.
* Graded version: `labs/part12/week45_golive` (`rollout_decision`, `capital_ramp`, `session_times`, `premarket_check`).